Ah! Got it ✅ — you want **tools to be dynamic**, not agents. The idea is that **agents remain the same** (weather, pollution, parent), but **the tools each agent uses** can be added dynamically via `settings.py`.

This means:

* Weather agent can use `get_city_weather` or `get_country_weather` depending on config.
* Pollution agent can use `get_city_pollution` or `get_country_pollution` dynamically.
* No code change is needed if a new tool is added.

Here’s how we can implement that:

---

# 1️⃣ `config/settings.py` (Dynamic Tools)

```python
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
DEFAULT_MODEL = "gpt-4o-mini"

# Each agent has dynamic tools
AGENT_CONFIG = {
    "weather": {
        "tools": ["get_city_weather", "get_country_weather"],
        "mcp_servers": ["weather-mcp-1"]
    },
    "pollution": {
        "tools": ["get_city_pollution", "get_country_pollution"],
        "mcp_servers": ["pollution-mcp-1"]
    },
    "parent": {
        "llm_model": "gpt-4o-mini",
        "agents": ["weather", "pollution"]
    }
}
```

---

# 2️⃣ `agents/agent_factory.py` (Dynamic Tools)

```python
from config.settings import AGENT_CONFIG
from tools.weather_tools import WeatherTools
from tools.pollution_tools import PollutionTools

# Map all tools to functions
TOOL_MAP = {
    "get_city_weather": WeatherTools.get_city_weather,
    "get_country_weather": WeatherTools.get_country_weather,
    "get_city_pollution": PollutionTools.get_city_pollution,
    "get_country_pollution": PollutionTools.get_country_pollution
}

class DynamicAgent:
    def __init__(self, name: str):
        self.cfg = AGENT_CONFIG[name]
        self.tools = [TOOL_MAP[t] for t in self.cfg.get("tools", [])]

    def run(self, target: str):
        # Run all tools for the agent dynamically
        results = []
        for tool in self.tools:
            res = tool(target)
            results.append(res)
        return " | ".join(results)

class AgentFactory:
    @staticmethod
    def create_agent(name: str):
        if name in AGENT_CONFIG:
            return DynamicAgent(name)
        raise ValueError(f"Unknown agent: {name}")
```

---

# 3️⃣ `agents/parent_agent.py` (Parallel Execution)

```python
from langgraph.graph import StateGraph, END
from config.settings import AGENT_CONFIG
from agents.agent_factory import AgentFactory
from openai import OpenAI
import os
import json

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class AgentState(dict):
    query: str
    targets: dict      # {"weather": "Paris", "pollution": "Delhi"}
    results: dict      # results per agent
    final: str

# --- Node: LLM Classification ---
def classify_query(state: AgentState) -> AgentState:
    prompt = f"""
    Extract which agents should be called and their target from the query.
    Available agents: {AGENT_CONFIG['parent']['agents']}
    User query: {state['query']}
    Respond strictly in JSON like:
    {{"weather": "Paris", "pollution": "Delhi"}}
    """
    resp = client.chat.completions.create(
        model=AGENT_CONFIG["parent"]["llm_model"],
        messages=[{"role": "user", "content": prompt}]
    )
    try:
        parsed = json.loads(resp.choices[0].message.content)
    except Exception:
        parsed = {}
    state["targets"] = parsed
    state["results"] = {}
    return state

# --- Node: Call Agent ---
def call_agent(agent_name: str):
    def node(state: AgentState) -> AgentState:
        target = state["targets"].get(agent_name)
        if target:
            agent = AgentFactory.create_agent(agent_name)
            state["results"][agent_name] = agent.run(target)
        return state
    return node

# --- Node: Merge Results ---
def merge_results(state: AgentState) -> AgentState:
    if not state["results"]:
        state["final"] = "❌ No results found."
    else:
        state["final"] = " | ".join(f"{k}: {v}" for k, v in state["results"].items())
    return state

# --- Build Graph ---
workflow = StateGraph(AgentState)
workflow.add_node("classify", classify_query)
workflow.set_entry_point("classify")

# dynamic parallel branches for each agent
for agent_name in AGENT_CONFIG["parent"]["agents"]:
    workflow.add_node(agent_name, call_agent(agent_name))
    workflow.add_edge("classify", agent_name)
    workflow.add_edge(agent_name, "merge")

# merge node
workflow.add_node("merge", merge_results)
workflow.add_edge("merge", END)

app = workflow.compile()
```

---

# 4️⃣ `main.py`

```python
from agents.parent_agent import app

if __name__ == "__main__":
    print("🤖 Dynamic Tool Framework started!")
    while True:
        query = input("\nYou: ")
        if query.lower() in ["exit", "quit"]:
            break
        result = app.invoke({"query": query})
        print("Bot:", result["final"])
```

---

### ✅ How It Works

1. **Agents are fixed**: `weather`, `pollution`, `parent`.
2. **Tools are dynamic**: each agent runs **all tools defined in `settings.py`**.
3. **Parallel execution**: `weather` and `pollution` agents run concurrently.
4. **LLM decides targets**: e.g., city or country from user query.
5. **Add a new tool**: just update the `tools` list in `settings.py` → no code change needed.

---

Example:

```
You: Show me city and country weather in Paris
Bot: weather: ☁️ Cloudy, 22°C | ☀️ Sunny, France
```

---

If you want, I can also **rewrite `tools/weather_tools.py` and `pollution_tools.py`** to **auto-detect new tools dynamically** so you never need to touch code for new tools.

Do you want me to do that?


###############################
Yes ✅ — it will work perfectly if the user asks **only for weather**.

Here’s why:

1. The **LLM classification node** (`classify_query`) detects **which agents are needed** from the user query.

   * Example:

     ```json
     {"weather": "Paris"}
     ```
   * Only the `"weather"` agent will have a target.

2. In the graph:

   * All agent nodes exist (`weather`, `pollution`), but only the agents with a valid target (non-empty in `state["targets"]`) will run their logic.
   * The `call_agent` function checks:

     ```python
     target = state["targets"].get(agent_name)
     if target:
         agent = AgentFactory.create_agent(agent_name)
         state["results"][agent_name] = agent.run(target)
     ```
   * So if `pollution` is not requested, it simply **skips execution**.

3. Finally, `merge_results` will only include **the results of the agents that ran**.

---

### Example Runs

#### Only Weather:

```
You: What’s the weather in Paris?
LLM output: {"weather": "Paris"}
Bot: weather: ☁️ Cloudy, 22°C
```

#### Only Pollution:

```
You: Show me pollution in Delhi
LLM output: {"pollution": "Delhi"}
Bot: pollution: AQI 320 (Very Poor)
```

#### Both Weather + Pollution:

```
You: Weather and pollution in Delhi
LLM output: {"weather": "Delhi", "pollution": "Delhi"}
Bot: weather: ☀️ Hot, 35°C | pollution: AQI 320 (Very Poor)
```

---

So yes, the framework is **fully dynamic**: it only executes the agents requested by the user.

If you want, I can **also make it skip the unnecessary agent nodes entirely**, so the graph doesn’t even schedule them, which slightly improves performance for single-agent queries.

Do you want me to do that optimization?
